<a href="https://colab.research.google.com/github/xmayksx1/DAX-BI/blob/main/Auto_OC_LOREAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

carga = r"C:\Users\Usuario\OneDrive - Aceitera El Real\Escritorio\Planificación - Planificación y Control\35. Dash Temporal 2026\Prueba de Automatización\Flujos de Trabajo\Drop Aceitera.XLSX"
guardado = r"C:\Users\Usuario\OneDrive - Aceitera El Real\Escritorio\Planificación - Planificación y Control\35. Dash Temporal 2026\Prueba de Automatización\Flujos de Trabajo\Resumen_Facturacion.xlsx"
ean = r"C:\Users\Usuario\OneDrive - Aceitera El Real\Escritorio\Planificación - Planificación y Control\35. Dash Temporal 2026\Prueba de Automatización\Flujos de Trabajo\Codigos EAN_LN.xlsx"


df = pd.read_excel(carga)

OC = df[["Código EAN/UPC", "Ctd.facturada", "Precio Crédito Unitario"]].copy()
OC["Monto"] = OC["Ctd.facturada"] * OC["Precio Crédito Unitario"]

OCG = OC.groupby("Código EAN/UPC", as_index=False).agg({
    "Ctd.facturada": "sum",
    "Precio Crédito Unitario": "mean",
    "Monto": "sum"
})

df_ean = pd.read_excel(ean, header = 1)
ean_link= df_ean[["EAN", "LN"]]

df_duplicados = OC[OC.duplicated(subset=["Código EAN/UPC"], keep=False)].copy()
df_duplicados = df_duplicados.sort_values(by=["Código EAN/UPC", "Precio Crédito Unitario"])

with pd.ExcelWriter(guardado) as writer:
    OCG.to_excel(writer, sheet_name='Resumen_Consolidado', index=False)
    if not df_duplicados.empty:
        df_duplicados.to_excel(writer, sheet_name='Revision_Duplicados', index=False)
    else:
        pd.DataFrame({"Mensaje": ["No hay códigos duplicados"]}).to_excel(writer, sheet_name='Revision_Duplicados', index=False)

OCG_final = pd.merge(
    OCG,
    ean_link,
    left_on="Código EAN/UPC",
    right_on="EAN",
    how="left"
).drop(columns=["EAN"])


df_duplicados_final = pd.merge(
    df_duplicados,
    ean_link,
    left_on="Código EAN/UPC",
    right_on="EAN",
    how="left"
).drop(columns=["EAN"])


with pd.ExcelWriter(guardado) as writer:
    OCG_final.to_excel(writer, sheet_name='Resumen_Consolidado', index=False)

    if not df_duplicados_final.empty:
        df_duplicados_final.to_excel(writer, sheet_name='Revision_Duplicados', index=False)
    else:
        pd.DataFrame({"Mensaje": ["No hay códigos duplicados"]}).to_excel(writer, sheet_name='Revision_Duplicados', index=False)

print("Proceso completado. Se integró la información de LN en ambas hojas.")

display(ean_link)
print("Proceso completado. El archivo con ambas hojas se guardó correctamente.")